# AgentRegistry on kind + AWS Bedrock AgentCore

Drive the enterprise **`arctl`** CLI through the whole agent lifecycle, then deploy **one** published agent to **two** runtimes from the same catalog: Solo Enterprise for **kagent** in a local kind cluster, and **AWS Bedrock AgentCore**. The model is **Anthropic Claude** (`claude-haiku-4-5`) in both places.

```mermaid
flowchart LR
  Eng[Solo engineer] -->|arctl init / build / apply| Daemon[arctl daemon<br/>catalog + UI :12121]
  Daemon --> Cat[(catalog<br/>summarizer agent<br/>textkit MCP<br/>summary-style skill)]
  Daemon -->|Runtime: Kubernetes| Kagent[kagent on kind<br/>OIDC via Keycloak]
  Daemon -->|Runtime: BedrockAgentCore| AC[AWS Bedrock<br/>AgentCore]
  Cat -. one agent .-> Kagent
  Cat -. same agent .-> AC
  classDef cp fill:#7C3AED,color:#fff,stroke:#5B21B6,stroke-width:2px
  classDef rt fill:#D1FAE5,stroke:#10B981,color:#064E3B,stroke-width:2px
  class Daemon,Cat cp
  class Kagent,AC rt
```

**What you need:** Docker, and the prerequisites the next cells install. Secrets stay out of this notebook and out of git — you put them once in a gitignored `.env.local`. This notebook uses a **bash kernel** ([`bash_kernel`](https://github.com/takluyver/bash_kernel)); every cell is shell.

> The kind + kagent bring-up takes ~15 minutes the first time (image pulls + the enterprise kagent install). The AgentCore section is an optional add-on that needs an AWS account.

## 0. Credentials — secure, but simple

No keys go in notebook cells (they'd be committed). Instead they live in a gitignored **`.env.local`** you fill in once. The cell below creates it from `env.example` on first run — **edit `.env.local`, then re-run this cell.**

- `ANTHROPIC_API_KEY` and `SOLO_LICENSE_KEY` are required for the kagent path.
- `AWS_PROFILE` / `AWS_REGION` are only needed for the AgentCore add-on.
- Already keep Solo creds in an existing env file? Set `SECRETS_FILE=/path/to/it` in `.env.local` and it's sourced too.

The cell prints only whether each value is **set** (and its length) — never the value itself.

In [ ]:
# Run from this notebook's folder; create .env.local on first run.
if [ ! -f .env.local ]; then
  cp env.example .env.local
  echo "Created .env.local from env.example."
  echo "==> Edit .env.local with your keys, then RE-RUN this cell."
fi

set -a
[ -f .env.local ] && . ./.env.local
[ -n "${SECRETS_FILE:-}" ] && [ -f "$SECRETS_FILE" ] && . "$SECRETS_FILE"
set +a

# arctl is installed under ~/.arctl/bin by the prereqs step.
export PATH="$HOME/.arctl/bin:$PATH"
export CLUSTER_NAME="${CLUSTER_NAME:-agentcore-demo}"
# The standalone daemon pulls the public server image and uses its embedded IdP.
export DOCKER_REPO="${DOCKER_REPO:-solo-public/agentregistry-enterprise}"
export OIDC_AUTO_AUTH_ENABLED="${OIDC_AUTO_AUTH_ENABLED:-true}"

mask() { if [ -n "$1" ]; then echo "set (${#1} chars)"; else echo "MISSING"; fi; }
echo "ANTHROPIC_API_KEY : $(mask "${ANTHROPIC_API_KEY:-}")"
echo "SOLO_LICENSE_KEY  : $(mask "${SOLO_LICENSE_KEY:-}")"
echo "AWS_PROFILE       : ${AWS_PROFILE:-<unset> (only needed for AgentCore)}"
echo "AWS_REGION        : ${AWS_REGION:-us-east-1}"
echo "arctl             : $(arctl version 2>/dev/null | awk '/arctl version/{print $3}' || echo '<not installed yet — run step 1>')"
echo "cluster           : kind-$CLUSTER_NAME"

## 1. Prerequisites

Install/validate the tooling (kind, kubectl, helm, jq, gh, uv, aws, gcloud) and the **enterprise `arctl` pinned to `v2026.5.4`** — the latest version that still ships the local `daemon` (v2026.6.x moved the registry server onto a cluster). On macOS missing CLIs are installed with Homebrew; otherwise it tells you what to install.

In [ ]:
./scripts/00-prereqs.sh

## 2. Bring up the local platform

Four idempotent steps (each is its own script so you can see the seams):

1. **`01-cluster.sh`** — kind cluster `agentcore-demo` + a host-side OCI registry on `localhost:5001` + Gateway API CRDs.
2. **`02-keycloak.sh`** — Keycloak with the `solo` realm; the enterprise kagent controller validates OIDC against it.
3. **`03-kagent.sh`** — Solo Enterprise for kagent (Anthropic as the default provider), wired to Keycloak. *(~10 min — image pulls + install.)*
4. **`04-daemon.sh`** — the local `arctl daemon` (catalog + UI on `http://localhost:12121`) and a bearer token for `arctl`.

Re-running is safe; finished steps short-circuit.

In [ ]:
./scripts/01-cluster.sh && ./scripts/02-keycloak.sh && ./scripts/03-kagent.sh && ./scripts/04-daemon.sh

## 3. Scaffold the building blocks with `arctl init`

The demo ships three artifact projects, already scaffolded **and** customized, under `artifacts/`:

| Kind | Name | What it is |
|------|------|-----------|
| MCPServer | `acme/textkit` | two tools: `word_count`, `extract_links` (FastMCP, Python) |
| Skill | `summary-style` | a `SKILL.md` house format, baked into the agent at build |
| Agent | `summarizer` | ADK Python, **Anthropic `claude-haiku-4-5`**, uses textkit + the skill |

They were generated with the commands below. Run this cell to watch `arctl init` produce a fresh agent from scratch in a scratch dir (the storyboard's "new agent project" beat) — then we use the committed, customized versions for the rest.

In [ ]:
# The exact commands that produced artifacts/ (customized afterwards with real tools + SKILL.md):
cat <<'CMDS'
  arctl init mcp   acme/textkit  --framework fastmcp --language python
  arctl init skill summary-style
  arctl init agent summarizer    --framework adk --language python \
      --model-provider anthropic --model-name claude-haiku-4-5 \
      --local-mcp ./textkit
CMDS

# Demonstrate scaffolding live into a throwaway dir (does not touch artifacts/):
rm -rf /tmp/arctl-scratch && mkdir -p /tmp/arctl-scratch
( cd /tmp/arctl-scratch && arctl init agent demoagent \
    --framework adk --language python \
    --model-provider anthropic --model-name claude-haiku-4-5 >/dev/null )
echo "--- arctl generated: ---"
find /tmp/arctl-scratch/demoagent -maxdepth 2 -type f | sed 's#/tmp/arctl-scratch/##' | sort

echo; echo "--- the demo's committed, customized artifacts: ---"
./scripts/05-scaffold.sh

## 4. Prove it locally with `arctl run` (no cluster)

Before any cluster, `arctl run` is the inner dev loop. `test-local.sh` starts the textkit MCP on the host, then `arctl run` builds the agent image, waits for its endpoint, and drops you into an **interactive A2A chat** — the full agent → tools path on Docker alone.

This cell is interactive, so run it in a terminal rather than inline:

```sh
./scripts/test-local.sh
# then type:  summarize this: <paste a paragraph with a couple of https:// links>
# Ctrl-C to exit
```

## 5. Build the images and publish to the catalog

`arctl build --push` builds the textkit MCP and the summarizer agent images and pushes them to `localhost:5001`; `arctl apply` publishes all three artifacts to the catalog. Order matters once: the MCPServer must exist before the Agent that references it.

In [ ]:
./scripts/06-build-publish.sh

In [ ]:
# The catalog is now the shared source of truth:
arctl get mcp acme/textkit; arctl get skill summary-style; arctl get agent summarizer

## 6. Deploy the agent onto kagent (runtime #1)

A **Runtime** points the registry at somewhere agents can run; a **Deployment** binds an Agent to a Runtime. `07-runtime-deploy.sh` registers a `Kubernetes` Runtime (`kind-kagent`) pointing at the cluster, then applies a Deployment. The registry's Kubernetes adapter translates it into kagent CRDs, and the controller schedules the agent (BYO) and the textkit MCP (kmcp).

In [ ]:
./scripts/07-runtime-deploy.sh

In [ ]:
kubectl --context kind-$CLUSTER_NAME -n kagent get agents,pods

## 7. Talk to the hosted agent

The agent sits behind Solo Enterprise for kagent's OIDC interceptor, so the test is honest: `ask.sh` mints a real Keycloak token for `alice` (group `field-fte` → kagent Admin) and sends an A2A `message/send` over the controller's endpoint. The reply comes from the model, using the textkit tools and the house-style skill.

In [ ]:
./scripts/ask.sh "summarize this: AgentRegistry is an open catalog for agents, MCP servers and skills. arctl scaffolds an artifact, builds it into an OCI image, and publishes it so others can reuse it. Docs at https://aregistry.ai and source at https://github.com/agentregistry-dev/agentregistry."

Optional: open the kagent dashboard.

```sh
./scripts/port-forward.sh   # http://localhost:8080
```

---
# AWS Bedrock AgentCore add-on (runtime #2)

The same published `summarizer` agent, deployed a second time — this time to AWS Bedrock AgentCore. Only the Deployment's `runtimeRef` changes. This part needs an AWS account; skip it for the local-only demo.

## 8. Sign in to AWS

Set `AWS_PROFILE` in `.env.local` (re-run the Setup cell), then sign in. `aws sso login` opens your browser; the rest of the section reuses the session. Nothing here prints your account or role.

Run the login in a terminal if your browser doesn't pop from the notebook:

```sh
aws sso login --profile "$AWS_PROFILE"
```

In [ ]:
if [ -z "${AWS_PROFILE:-}" ]; then
  echo "Set AWS_PROFILE in .env.local and re-run the Setup cell first."
else
  aws sts get-caller-identity >/dev/null 2>&1 || aws sso login --profile "$AWS_PROFILE"
  # Confirm we have a session (account masked):
  aws sts get-caller-identity --query '{account: Account, region: '"'""${AWS_REGION:-us-east-1}""'"'}' --output json \
    | sed -E 's/[0-9]{12}/<account-id>/'
fi

## 9. Deploy the same agent to AgentCore

`08-agentcore.sh` does the whole AWS side from the local daemon:

1. `arctl runtime setup bedrock-agent-core` → a CloudFormation template granting AgentRegistry a cross-account role, plus an External ID.
2. Deploys that stack and reads back the role ARN.
3. Registers a **`BedrockAgentCore`** Runtime (`aws-agentcore`).
4. Pushes the agent image to **ECR** (AgentCore can't pull `localhost:5001`).
5. Applies a Deployment binding `summarizer` → `aws-agentcore`, carrying `ANTHROPIC_API_KEY`.

> AgentCore clones the agent **source** from git at deploy time. By default that's this repo + branch; the branch must be pushed somewhere AWS can reach it. Override with `AGENT_GIT_URL` / `AGENT_GIT_BRANCH` / `AGENT_GIT_SUBFOLDER` in `.env.local` if needed.

In [ ]:
./scripts/08-agentcore.sh

In [ ]:
# Watch the AgentCore deployment reconcile:
arctl get deployments

## 10. Test the agent on AgentCore

Open **Amazon Bedrock → AgentCore** in the AWS console, find your runtime, and send a JSON-RPC `message/send` payload in the playground:

```json
{
  "jsonrpc": "2.0",
  "id": "req-001",
  "method": "message/send",
  "params": {
    "message": {
      "role": "user",
      "messageId": "12345678-1234-1234-1234-123456789012",
      "parts": [{"kind": "text", "text": "summarize this: <paste a paragraph with a couple of https:// links>"}]
    }
  }
}
```

Same catalog agent, same model, now running in AWS — the only thing that changed between runtime #1 and #2 was the Deployment's `runtimeRef`.

## 11. Teardown

`cleanup.sh` removes the AWS AgentCore bits (CloudFormation stack, runtime, ECR repo, deployment) and the local platform (kind cluster, arctl daemon, local registry). The `agentcore`-only and `all` forms are both safe to re-run.

```sh
./scripts/cleanup.sh agentcore   # AWS only
./scripts/cleanup.sh             # everything
```

In [ ]:
# Uncomment to tear everything down:
# ./scripts/cleanup.sh